# Company financials and macroeconomic conditions

Read data/company_financials.csv and data/macro_data.csv, and write data/company_macro_panel.csv. Each row represents one company and fiscal year. Summarize macro observations over that row's period_start through period_end.

This notebook uses saved CSV files only. It makes no SEC or FRED requests. The result is a descriptive fiscal-period panel, not a point-in-time forecasting dataset.

The shape and coverage checks below refer to the fixed five-company, 2021-2025 teaching snapshot. Review them when deliberately refreshing its inputs; do not force new data to reproduce old counts.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path, *current_path.parents]

    for candidate_path in candidate_paths:
        data_dir = candidate_path / "data"
        src_dir = candidate_path / "src"

        if data_dir.is_dir() and src_dir.is_dir():
            return candidate_path

    raise FileNotFoundError(
        "Could not find the project root. "
        "Run the notebook from within the project directory."
    )


PROJECT_ROOT = find_project_root()
COMPANY_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "company_financials.csv"
)
MACRO_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "macro_data.csv"
)
PANEL_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "company_macro_panel.csv"
)

EXPECTED_TICKERS = [
    "AAPL",
    "MSFT",
    "JPM",
    "TSLA",
    "WMT",
]
EXPECTED_COMPANY_SHAPE = (25, 39)
EXPECTED_PANEL_SHAPE = (25, 56)

COMPANY_REQUIRED_COLUMNS = [
    "company_name",
    "ticker",
    "cik",
    "sector",
    "fiscal_year",
    "period_start",
    "period_end",
    "balance_sheet_date",
    "revenue_usd",
    "net_income_usd",
    "assets_usd",
    "liabilities_usd",
    "operating_cash_flow_usd",
    "source",
]
COMPANY_DATE_COLUMNS = [
    "period_start",
    "period_end",
    "balance_sheet_date",
    "revenue_filed",
    "net_income_filed",
    "assets_filed",
    "liabilities_filed",
    "operating_cash_flow_filed",
]

MONTHLY_SERIES_IDS = [
    "UNRATE",
    "FEDFUNDS",
    "CPIAUCSL",
]
DAILY_SERIES_IDS = [
    "DGS10",
    "T10Y2Y",
]
MACRO_SERIES_COLUMNS = [
    *MONTHLY_SERIES_IDS,
    *DAILY_SERIES_IDS,
]
MACRO_REQUIRED_COLUMNS = [
    "date",
    *MACRO_SERIES_COLUMNS,
]

MACRO_FEATURE_COLUMNS = [
    "fiscal_year_end_month",
    "fiscal_year_spans_two_calendar_years",
    "macro_window_start_month",
    "macro_window_end_month",
    "macro_expected_month_count",
    "macro_observed_month_count",
    "macro_missing_value_count",
    "macro_coverage_ratio",
    "macro_coverage_status",
    "unrate_mean_pct",
    "fedfunds_mean_pct",
    "cpi_start_index",
    "cpi_end_index",
    "cpi_change_pct",
    "dgs10_mean_pct",
    "t10y2y_mean_pct",
    "macro_source",
]

MACRO_VALUE_FEATURE_COLUMNS = [
    "unrate_mean_pct",
    "fedfunds_mean_pct",
    "cpi_start_index",
    "cpi_end_index",
    "cpi_change_pct",
    "dgs10_mean_pct",
    "t10y2y_mean_pct",
]

print("project_root:", PROJECT_ROOT)
print("company_csv_path:", COMPANY_CSV_PATH)
print("macro_csv_path:", MACRO_CSV_PATH)
print("panel_csv_path:", PANEL_CSV_PATH)

In [ ]:
if not COMPANY_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Company financials CSV not found: {COMPANY_CSV_PATH}"
    )

if not MACRO_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Macro data CSV not found: {MACRO_CSV_PATH}"
    )

company_df = pd.read_csv(
    COMPANY_CSV_PATH,
    dtype={"cik": "string"},
)
macro_df = pd.read_csv(MACRO_CSV_PATH)

missing_company_columns = sorted(
    set(COMPANY_REQUIRED_COLUMNS) - set(company_df.columns)
)
if missing_company_columns:
    raise KeyError(
        "Company financials are missing required columns: "
        f"{missing_company_columns}"
    )

if list(macro_df.columns) != MACRO_REQUIRED_COLUMNS:
    raise ValueError(
        "The macro CSV columns do not match the expected schema: "
        f"{list(macro_df.columns)}"
    )

for date_column in COMPANY_DATE_COLUMNS:
    if date_column in company_df.columns:
        company_df[date_column] = pd.to_datetime(
            company_df[date_column],
            errors="coerce",
        )

company_df["fiscal_year"] = pd.to_numeric(
    company_df["fiscal_year"],
    errors="coerce",
)
if company_df["fiscal_year"].isna().any():
    raise ValueError("fiscal_year contains values that could not be converted to numbers.")
company_df["fiscal_year"] = company_df["fiscal_year"].astype(int)

company_df["ticker"] = (
    company_df["ticker"]
    .astype("string")
    .str.strip()
    .str.upper()
)
company_df["cik"] = (
    company_df["cik"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(10)
)

macro_df["date"] = pd.to_datetime(
    macro_df["date"],
    errors="coerce",
)
for series_id in MACRO_SERIES_COLUMNS:
    macro_df[series_id] = pd.to_numeric(
        macro_df[series_id],
        errors="coerce",
    )

if company_df.shape != EXPECTED_COMPANY_SHAPE:
    raise ValueError(
        "The company financials shape differs from the teaching snapshot: "
        f"{company_df.shape}"
    )

if set(company_df["ticker"].dropna()) != set(EXPECTED_TICKERS):
    raise ValueError(
        "The company financials contain unexpected tickers: "
        f"{sorted(company_df['ticker'].dropna().unique())}"
    )

company_duplicate_count = company_df.duplicated(
    subset=["ticker", "fiscal_year"]
).sum()
if company_duplicate_count != 0:
    raise ValueError(
        "Duplicate ticker-fiscal_year keys: "
        f"{company_duplicate_count}"
    )

company_key_date_missing_count = company_df[
    ["period_start", "period_end", "balance_sheet_date"]
].isna().sum().sum()
if company_key_date_missing_count != 0:
    raise ValueError(
        "Missing financial reporting dates: "
        f"{company_key_date_missing_count}"
    )

if not company_df["cik"].str.fullmatch(r"\d{10}").all():
    raise ValueError("CIK values must be ten-digit strings.")

if macro_df["date"].isna().any():
    raise ValueError("The macro CSV contains invalid dates.")

if macro_df["date"].duplicated().any():
    raise ValueError("The macro CSV contains duplicate dates.")

macro_df = macro_df.sort_values("date").reset_index(drop=True)

print("company_shape:", company_df.shape)
print("company_tickers:", sorted(company_df["ticker"].unique()))
print(
    "company_fiscal_year_range:",
    company_df["fiscal_year"].min(),
    company_df["fiscal_year"].max(),
)
print("macro_shape:", macro_df.shape)
print("macro_first_date:", macro_df["date"].min().date())
print("macro_last_date:", macro_df["date"].max().date())
print("macro_missing_summary:")
print(macro_df[MACRO_SERIES_COLUMNS].isna().sum())

In [ ]:
def build_monthly_macro(macro_data):
    working_df = macro_data.copy()
    working_df["month"] = working_df["date"].dt.to_period("M")

    monthly_parts = []

    for series_id in MONTHLY_SERIES_IDS:
        monthly_observations = working_df.loc[
            working_df[series_id].notna(),
            ["date", "month", series_id],
        ].copy()

        duplicate_month_count = (
            monthly_observations
            .groupby("month")
            .size()
            .gt(1)
            .sum()
        )
        if duplicate_month_count != 0:
            raise ValueError(
                f"{series_id} contains multiple observations in a calendar month: "
                f"{duplicate_month_count}"
            )

        monthly_series = (
            monthly_observations
            .sort_values("date")
            .groupby("month")[series_id]
            .last()
        )
        monthly_parts.append(monthly_series)

    for series_id in DAILY_SERIES_IDS:
        daily_observations = working_df.loc[
            working_df[series_id].notna(),
            ["month", series_id],
        ].copy()

        monthly_series = (
            daily_observations
            .groupby("month")[series_id]
            .mean()
        )
        monthly_parts.append(monthly_series)

    monthly_macro_df = (
        pd.concat(monthly_parts, axis=1)
        .sort_index()
        .reset_index()
    )

    monthly_macro_df["month_start"] = (
        monthly_macro_df["month"]
        .dt.to_timestamp(how="start")
    )
    monthly_macro_df["month_end"] = (
        monthly_macro_df["month"]
        .dt.to_timestamp(how="end")
        .dt.normalize()
    )

    monthly_macro_df = monthly_macro_df[
        [
            "month",
            "month_start",
            "month_end",
            *MACRO_SERIES_COLUMNS,
        ]
    ]

    return monthly_macro_df


monthly_macro_df = build_monthly_macro(macro_df)

expected_months = pd.period_range(
    start=monthly_macro_df["month"].min(),
    end=monthly_macro_df["month"].max(),
    freq="M",
)

if list(monthly_macro_df["month"]) != list(expected_months):
    raise ValueError("The monthly macro table has gaps in its calendar months.")

if monthly_macro_df["month"].duplicated().any():
    raise ValueError("The monthly macro table contains duplicate months.")

print("monthly_macro_shape:", monthly_macro_df.shape)
print("monthly_macro_first_month:", monthly_macro_df["month"].min())
print("monthly_macro_last_month:", monthly_macro_df["month"].max())
print("monthly_macro_missing_summary:")
print(monthly_macro_df[MACRO_SERIES_COLUMNS].isna().sum())

display(monthly_macro_df.head())

In [ ]:
def expected_months_for_period(period_start, period_end):
    if pd.isna(period_start) or pd.isna(period_end):
        raise ValueError("The fiscal-period start or end date is missing.")

    if period_start > period_end:
        raise ValueError("period_start must not be later than period_end.")

    candidate_months = pd.period_range(
        start=period_start.to_period("M"),
        end=period_end.to_period("M"),
        freq="M",
    )
    candidate_month_starts = candidate_months.to_timestamp(
        how="start"
    )

    inside_period_mask = (
        (candidate_month_starts >= period_start.normalize())
        & (candidate_month_starts <= period_end.normalize())
    )
    expected_months = candidate_months[inside_period_mask]

    if len(expected_months) == 0:
        raise ValueError("No calendar-month starts fall within the fiscal period.")

    return expected_months


sample_row = (
    company_df
    .loc[
        (company_df["ticker"] == "AAPL")
        & (company_df["fiscal_year"] == 2024)
    ]
    .iloc[0]
)

sample_expected_months = expected_months_for_period(
    period_start=sample_row["period_start"],
    period_end=sample_row["period_end"],
)

print("sample_ticker:", sample_row["ticker"])
print("sample_fiscal_year:", sample_row["fiscal_year"])
print("sample_period_start:", sample_row["period_start"].date())
print("sample_period_end:", sample_row["period_end"].date())
print("sample_expected_month_count:", len(sample_expected_months))
print("sample_first_month:", sample_expected_months[0])
print("sample_last_month:", sample_expected_months[-1])

In [ ]:
def mean_or_nan(dataframe, column_name):
    valid_values = dataframe[column_name].dropna()

    if valid_values.empty:
        return np.nan

    return valid_values.mean()


def summarize_macro_window(company_row, monthly_macro_data):
    period_start = company_row["period_start"]
    period_end = company_row["period_end"]

    expected_months = expected_months_for_period(
        period_start=period_start,
        period_end=period_end,
    )

    window_df = (
        monthly_macro_data
        .loc[
            monthly_macro_data["month"].isin(expected_months)
        ]
        .sort_values("month")
        .reset_index(drop=True)
    )

    expected_month_count = len(expected_months)
    observed_month_count = len(window_df)
    missing_value_count = int(
        window_df[MACRO_SERIES_COLUMNS]
        .isna()
        .sum()
        .sum()
    )
    coverage_ratio = (
        observed_month_count / expected_month_count
    )

    if observed_month_count == 0:
        coverage_status = "missing"
    elif (
        observed_month_count == expected_month_count
        and missing_value_count == 0
    ):
        coverage_status = "complete"
    else:
        coverage_status = "partial"

    cpi_values = window_df["CPIAUCSL"].dropna()

    if cpi_values.empty:
        cpi_start_index = np.nan
        cpi_end_index = np.nan
        cpi_change_pct = np.nan
    else:
        cpi_start_index = cpi_values.iloc[0]
        cpi_end_index = cpi_values.iloc[-1]

        if cpi_start_index == 0:
            cpi_change_pct = np.nan
        else:
            cpi_change_pct = (
                (cpi_end_index / cpi_start_index) - 1
            ) * 100

    return {
        "fiscal_year_end_month": int(period_end.month),
        "fiscal_year_spans_two_calendar_years": (
            period_start.year != period_end.year
        ),
        "macro_window_start_month": (
            expected_months[0].to_timestamp(how="start")
        ),
        "macro_window_end_month": (
            expected_months[-1].to_timestamp(how="start")
        ),
        "macro_expected_month_count": expected_month_count,
        "macro_observed_month_count": observed_month_count,
        "macro_missing_value_count": missing_value_count,
        "macro_coverage_ratio": coverage_ratio,
        "macro_coverage_status": coverage_status,
        "unrate_mean_pct": mean_or_nan(window_df, "UNRATE"),
        "fedfunds_mean_pct": mean_or_nan(
            window_df,
            "FEDFUNDS",
        ),
        "cpi_start_index": cpi_start_index,
        "cpi_end_index": cpi_end_index,
        "cpi_change_pct": cpi_change_pct,
        "dgs10_mean_pct": mean_or_nan(window_df, "DGS10"),
        "t10y2y_mean_pct": mean_or_nan(
            window_df,
            "T10Y2Y",
        ),
        "macro_source": "FRED",
    }

In [ ]:
macro_summary_records = []

for _, company_row in company_df.iterrows():
    summary_record = {
        "ticker": company_row["ticker"],
        "fiscal_year": int(company_row["fiscal_year"]),
    }
    summary_record.update(
        summarize_macro_window(
            company_row=company_row,
            monthly_macro_data=monthly_macro_df,
        )
    )
    macro_summary_records.append(summary_record)

macro_summary_df = pd.DataFrame(macro_summary_records)
macro_summary_df = macro_summary_df[
    [
        "ticker",
        "fiscal_year",
        *MACRO_FEATURE_COLUMNS,
    ]
]

macro_summary_df[
    [
        "macro_coverage_ratio",
        *MACRO_VALUE_FEATURE_COLUMNS,
    ]
] = macro_summary_df[
    [
        "macro_coverage_ratio",
        *MACRO_VALUE_FEATURE_COLUMNS,
    ]
].round(4)

if macro_summary_df.duplicated(
    subset=["ticker", "fiscal_year"]
).any():
    raise ValueError("The macro summary contains duplicate keys.")

print("macro_summary_shape:", macro_summary_df.shape)
print("macro_coverage_status_counts:")
print(
    macro_summary_df["macro_coverage_status"]
    .value_counts(dropna=False)
)

display(
    macro_summary_df.loc[
        macro_summary_df["fiscal_year"] == 2025,
        [
            "ticker",
            "fiscal_year",
            "macro_window_start_month",
            "macro_window_end_month",
            "macro_expected_month_count",
            "macro_observed_month_count",
            "macro_coverage_ratio",
            "macro_coverage_status",
        ],
    ]
)

In [ ]:
company_columns = list(company_df.columns)

panel_df = company_df.merge(
    macro_summary_df,
    on=["ticker", "fiscal_year"],
    how="left",
    validate="one_to_one",
)
panel_df = panel_df[
    [
        *company_columns,
        *MACRO_FEATURE_COLUMNS,
    ]
]

sort_order_map = {
    ticker: order
    for order, ticker in enumerate(EXPECTED_TICKERS)
}
panel_df["_ticker_order"] = panel_df["ticker"].map(
    sort_order_map
)
panel_df = (
    panel_df
    .sort_values(["_ticker_order", "fiscal_year"])
    .drop(columns="_ticker_order")
    .reset_index(drop=True)
)

if panel_df.shape != EXPECTED_PANEL_SHAPE:
    raise ValueError(
        "The merged panel shape differs from the teaching snapshot: "
        f"{panel_df.shape}"
    )

if panel_df.duplicated(
    subset=["ticker", "fiscal_year"]
).any():
    raise ValueError("The merged panel contains duplicate keys.")

if not panel_df["macro_coverage_ratio"].between(0, 1).all():
    raise ValueError("macro_coverage_ratio must be between zero and one.")

allowed_coverage_statuses = {
    "complete",
    "partial",
    "missing",
}
if not set(panel_df["macro_coverage_status"]).issubset(
    allowed_coverage_statuses
):
    raise ValueError("An unknown macro coverage status was found.")

complete_rows = panel_df.loc[
    panel_df["macro_coverage_status"] == "complete"
]
if not complete_rows["macro_coverage_ratio"].eq(1).all():
    raise ValueError("A complete row must have a coverage ratio of one.")

if not complete_rows["macro_missing_value_count"].eq(0).all():
    raise ValueError("A complete row contains missing macro values.")

missing_rows = panel_df.loc[
    panel_df["macro_coverage_status"] == "missing"
]
if not missing_rows[MACRO_VALUE_FEATURE_COLUMNS].isna().all().all():
    raise ValueError("A missing-coverage row contains a macro summary value.")

original_company_check = (
    company_df
    .sort_values(["ticker", "fiscal_year"])
    .reset_index(drop=True)
)
merged_company_check = (
    panel_df[company_columns]
    .sort_values(["ticker", "fiscal_year"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    original_company_check,
    merged_company_check,
    check_dtype=False,
)

coverage_status_counts = (
    panel_df["macro_coverage_status"]
    .value_counts()
    .to_dict()
)

if monthly_macro_df["month"].max() == pd.Period("2024-12"):
    expected_status_counts = {
        "complete": 20,
        "partial": 3,
        "missing": 2,
    }

    if coverage_status_counts != expected_status_counts:
        raise ValueError(
            "Coverage counts differ from the fixed teaching snapshot: "
            f"{coverage_status_counts}"
        )

print("panel_shape:", panel_df.shape)
print("panel_column_count:", len(panel_df.columns))
print("panel_duplicate_key_count:", panel_df.duplicated(
    subset=["ticker", "fiscal_year"]
).sum())
print("company_columns_preserved:", True)
print("coverage_status_counts:", coverage_status_counts)

In [ ]:
PANEL_DATE_COLUMNS = [
    *COMPANY_DATE_COLUMNS,
    "macro_window_start_month",
    "macro_window_end_month",
]

temporary_path = PANEL_CSV_PATH.with_suffix(".csv.tmp")

panel_df.to_csv(
    temporary_path,
    index=False,
    date_format="%Y-%m-%d",
)
temporary_path.replace(PANEL_CSV_PATH)

saved_panel_df = pd.read_csv(
    PANEL_CSV_PATH,
    dtype={"cik": "string"},
    parse_dates=PANEL_DATE_COLUMNS,
)

if saved_panel_df.shape != EXPECTED_PANEL_SHAPE:
    raise ValueError(
        "The saved panel shape differs from the expected shape: "
        f"{saved_panel_df.shape}"
    )

if list(saved_panel_df.columns) != list(panel_df.columns):
    raise ValueError("The saved panel column order changed.")

if saved_panel_df.duplicated(
    subset=["ticker", "fiscal_year"]
).any():
    raise ValueError("The saved panel contains duplicate keys.")

if not saved_panel_df["cik"].str.fullmatch(r"\d{10}").all():
    raise ValueError("The saved panel contains invalid CIK values.")

print("saved_panel_path:", PANEL_CSV_PATH)
print("saved_panel_exists:", PANEL_CSV_PATH.exists())
print("saved_panel_shape:", saved_panel_df.shape)
print("saved_panel_columns:", list(saved_panel_df.columns))
print("saved_coverage_status_counts:")
print(
    saved_panel_df["macro_coverage_status"]
    .value_counts(dropna=False)
)

In [ ]:
SELECTED_TICKER = "AAPL"

analysis_df = (
    panel_df
    .loc[panel_df["ticker"] == SELECTED_TICKER]
    .sort_values("fiscal_year")
    .copy()
)
analysis_df["revenue_usd_billions"] = (
    analysis_df["revenue_usd"] / 1_000_000_000
)
analysis_df["net_income_usd_billions"] = (
    analysis_df["net_income_usd"] / 1_000_000_000
)

analysis_columns = [
    "ticker",
    "fiscal_year",
    "period_start",
    "period_end",
    "revenue_usd_billions",
    "net_income_usd_billions",
    "unrate_mean_pct",
    "fedfunds_mean_pct",
    "cpi_change_pct",
    "dgs10_mean_pct",
    "t10y2y_mean_pct",
    "macro_coverage_status",
]

display(analysis_df[analysis_columns])

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(
    analysis_df["fiscal_year"],
    analysis_df["revenue_usd_billions"],
    marker="o",
)
plt.title(f"{SELECTED_TICKER} Revenue by Fiscal Year")
plt.xlabel("Fiscal Year")
plt.ylabel("Revenue, USD Billions")
plt.tight_layout()
plt.show()

In [ ]:
complete_analysis_df = analysis_df.loc[
    analysis_df["macro_coverage_status"] == "complete"
].copy()

plt.figure(figsize=(9, 4.5))
plt.plot(
    complete_analysis_df["fiscal_year"],
    complete_analysis_df["unrate_mean_pct"],
    marker="o",
    label="UNRATE Mean",
)
plt.plot(
    complete_analysis_df["fiscal_year"],
    complete_analysis_df["fedfunds_mean_pct"],
    marker="o",
    label="FEDFUNDS Mean",
)
plt.title(
    f"{SELECTED_TICKER} Fiscal-Period Macro Averages"
)
plt.xlabel("Fiscal Year")
plt.ylabel("Percent")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fiscal_year_2024_periods = (
    panel_df
    .loc[
        panel_df["fiscal_year"] == 2024,
        [
            "ticker",
            "fiscal_year",
            "period_start",
            "period_end",
            "fiscal_year_end_month",
            "fiscal_year_spans_two_calendar_years",
            "macro_window_start_month",
            "macro_window_end_month",
            "macro_expected_month_count",
            "macro_coverage_status",
        ],
    ]
    .sort_values("ticker")
)

display(fiscal_year_2024_periods)